# 00 · Konversi CSV → Parquet (Polars)

Notebook ini dijalankan **sekali** di awal. Semua notebook berikutnya (audit, EDA, fitur, model) membaca file Parquet hasilnya.

**Kenapa Parquet + Polars?**
- **Tipe data terkunci**: tanggal sudah jadi `Datetime`, uang jadi `Int64`, tidak perlu `parse_dates` / `dtype=` lagi setiap baca.
- **Lebih kecil & jauh lebih cepat dibaca** daripada CSV (kolumnar + kompresi zstd).
- **Bisa baca sebagian kolom saja** (`columns=[...]`) dan **lazy** (`pl.scan_parquet`) → hemat RAM untuk tabel besar seperti e-commerce.

**Struktur folder yang diharapkan**
```
project/
├── data/
│   ├── raw/        ← semua CSV asli + DATA_DICTIONARY.md
│   └── parquet/    ← output notebook ini
└── notebooks/      ← notebook ini
```
Kalau struktur Anda berbeda, cukup ubah `RAW_DIR` dan `PQ_DIR` di sel konfigurasi.

In [1]:
from pathlib import Path
import json, time
import polars as pl

print("polars", pl.__version__)

# ===== KONFIGURASI =====
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PQ_DIR = ROOT / "data" / "parquet"

# File e-commerce: None = deteksi otomatis (prioritas file penuh `ecommerce_transactions.csv`,
# kalau tidak ada pakai file lain yang namanya diawali `ecommerce_transactions`, mis. hasil sampling).
ECOM_CSV = None
OVERWRITE = True
# =======================

PQ_DIR.mkdir(parents=True, exist_ok=True)
print("RAW_DIR:", RAW_DIR)
print("PQ_DIR :", PQ_DIR)

polars 2.0.0
RAW_DIR: d:\LOMBA\LOGIKA\data\raw
PQ_DIR : d:\LOMBA\LOGIKA\data\parquet


## 1. Skema eksplisit per tabel

Skema diambil dari `DATA_DICTIONARY.md`. Prinsipnya:
- **ID & kategori** → `String` (Parquet otomatis melakukan dictionary-encoding, jadi tetap kecil).
- **Uang** → `Int64` (rupiah utuh; `Int32` maksimal ±2,1 miliar sehingga tidak aman untuk saldo/utang).
- **Hitungan kecil** → `Int32`, **rasio/skor/bunga** → `Float64`.
- **Datetime** → dibaca sebagai teks lalu di-parse dengan format pasti (`strict=True`: kalau ada format aneh, konversi **gagal keras**, bukan diam-diam jadi null).
- Waktu disimpan **naive** (tanpa zona waktu) karena semuanya sudah waktu lokal Asia/Jakarta tanpa DST.

In [2]:
S, I64, I32, I8, F64 = pl.String, pl.Int64, pl.Int32, pl.Int8, pl.Float64
DT, D = "datetime", "date"          # penanda: di-parse setelah dibaca
DT_FMT, D_FMT = "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"

APP_COLS = {
    "application_id": S, "customer_id": S,
    "application_datetime": DT, "decision_datetime": DT,
    "loan_amount": I64, "term_months": I32, "annual_interest_rate": F64,
    "proposed_monthly_installment": I64, "loan_purpose": S,
}

SCHEMAS = {
    "train_applications": {**APP_COLS, "target": I8},
    "test_applications": APP_COLS,
    "sample_submission": {"application_id": S, "default_probability": F64},
    "customer_profiles": {
        "customer_id": S, "age": I32, "region": S, "employment_type": S,
        "employment_tenure_months": I32, "declared_monthly_income": I64,
        "dependents": I32, "housing_status": S,
    },
    "previous_applications": {
        "application_id": S, "customer_id": S,
        "application_datetime": DT, "decision_datetime": DT,
        "requested_amount": I64, "term_months": I32, "decision": S, "loan_id": S,
        "approved_amount": I64, "annual_interest_rate": F64, "scheduled_monthly_installment": I64,
    },
    "repayment_history": {
        "loan_id": S, "installment_id": S, "customer_id": S, "due_date": DT,
        "amount_due": I64, "amount_paid": I64, "last_payment_datetime": DT,
        "days_past_due": I32, "max_days_past_due": I32,
    },
    "account_behavior": {
        "customer_id": S, "month": D, "opening_balance": I64, "closing_balance": I64,
        "inflows": I64, "outflows": I64, "income_like_credits": I64,
        "average_balance": I64, "minimum_balance": I64, "low_balance_days": I32,
    },
    "bureau_observations": {
        "bureau_observation_id": S, "application_id": S, "customer_id": S, "bureau_provider": S,
        "report_datetime": DT, "available_datetime": DT, "score": F64,
        "active_accounts": I32, "outstanding_balance": I64, "revolving_credit_limit": I64,
        "revolving_outstanding_balance": I64, "utilization": F64,
        "inquiries_last_6m": I32, "delinquent_accounts_last_12m": I32,
    },
    "ecommerce_transactions": {
        "order_id": S, "order_item_id": S, "customer_id": S, "transaction_datetime": DT,
        "product_category": S, "quantity": I32, "item_total": I64, "order_total": I64,
        "payment_method": S, "order_status": S,
    },
}

# Urutan simpan: mengelompokkan baris per nasabah & waktu → kompresi lebih baik dan
# agregasi per nasabah nanti lebih cepat. Tabel aplikasi dibiarkan dalam urutan aslinya.
SORT_BY = {
    "previous_applications": ["customer_id", "application_datetime"],
    "repayment_history": ["customer_id", "loan_id", "due_date"],
    "account_behavior": ["customer_id", "month"],
    "bureau_observations": ["customer_id", "report_datetime", "bureau_provider"],
    "ecommerce_transactions": ["customer_id", "transaction_datetime", "order_id", "order_item_id"],
}

## 2. Cari file sumber

In [3]:
def find_ecom_csv():
    if ECOM_CSV is not None:
        return RAW_DIR / ECOM_CSV
    full = RAW_DIR / "ecommerce_transactions.csv"
    if full.exists():
        return full
    cands = sorted(RAW_DIR.glob("ecommerce_transactions*.csv"))
    if not cands:
        raise FileNotFoundError("Tidak ada file ecommerce_transactions*.csv di RAW_DIR")
    return cands[0]

SOURCES = {name: RAW_DIR / f"{name}.csv" for name in SCHEMAS if name != "ecommerce_transactions"}
SOURCES["ecommerce_transactions"] = find_ecom_csv()

missing = [str(p) for p in SOURCES.values() if not p.exists()]
assert not missing, f"File tidak ditemukan: {missing}"
for k, v in SOURCES.items():
    print(f"{k:<24} ← {v.name}  ({v.stat().st_size/1e6:,.1f} MB)")
ECOM_IS_SAMPLED = SOURCES["ecommerce_transactions"].name != "ecommerce_transactions.csv"
print("\nE-commerce versi:", "SAMPLING" if ECOM_IS_SAMPLED else "PENUH")

train_applications       ← train_applications.csv  (8.2 MB)
test_applications        ← test_applications.csv  (3.5 MB)
sample_submission        ← sample_submission.csv  (0.8 MB)
customer_profiles        ← customer_profiles.csv  (6.4 MB)
previous_applications    ← previous_applications.csv  (26.7 MB)
repayment_history        ← repayment_history.csv  (159.7 MB)
account_behavior         ← account_behavior.csv  (100.0 MB)
bureau_observations      ← bureau_observations.csv  (114.7 MB)
ecommerce_transactions   ← ecommerce_transactions.csv  (528.8 MB)

E-commerce versi: PENUH


## 3. Konversi

Memakai `scan_csv → sink_parquet` (*streaming*): file tidak perlu dimuat utuh ke RAM, aman untuk e-commerce versi penuh (4,36 juta baris).
Setiap tabel diverifikasi: header CSV harus sama persis dengan skema, dan jumlah baris Parquet harus sama dengan CSV.

In [4]:
def csv_header(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.readline().strip().lstrip("\ufeff").split(",")

def convert(name, src, schema):
    out = PQ_DIR / f"{name}.parquet"
    if out.exists() and not OVERWRITE:
        print(f"skip {name} (sudah ada)")
        return None
    header = csv_header(src)
    if header != list(schema):
        raise ValueError(f"{name}: header CSV {header} != skema {list(schema)}")

    read_schema = {c: (pl.String if t in (DT, D) else t) for c, t in schema.items()}
    parse = []
    for c, t in schema.items():
        if t == DT:
            parse.append(pl.col(c).str.to_datetime(DT_FMT, strict=True))
        elif t == D:
            parse.append(pl.col(c).str.to_date(D_FMT, strict=True))

    t0 = time.time()
    lf = pl.scan_csv(src, schema=read_schema)
    if parse:
        lf = lf.with_columns(parse)
    if name in SORT_BY:
        lf = lf.sort(SORT_BY[name])
    lf.sink_parquet(out, compression="zstd", statistics=True)

    n_csv = pl.scan_csv(src, schema=read_schema).select(pl.len()).collect().item()
    n_pq = pl.scan_parquet(out).select(pl.len()).collect().item()
    assert n_csv == n_pq, f"{name}: jumlah baris beda (csv={n_csv}, parquet={n_pq})"
    return {
        "table": name, "source": src.name, "rows": n_pq, "cols": len(schema),
        "csv_mb": round(src.stat().st_size / 1e6, 1),
        "parquet_mb": round(out.stat().st_size / 1e6, 1),
        "seconds": round(time.time() - t0, 1),
    }

report = []
for name, schema in SCHEMAS.items():
    r = convert(name, SOURCES[name], schema)
    if r:
        report.append(r)
        print(f"✓ {name:<24} {r['rows']:>10,} baris  {r['csv_mb']:>7.1f} MB → {r['parquet_mb']:>6.1f} MB  ({r['seconds']}s)")

report_df = pl.DataFrame(report).with_columns(
    (pl.col("parquet_mb") / pl.col("csv_mb")).round(3).alias("ratio")
)
report_df

✓ train_applications           70,000 baris      8.2 MB →    2.5 MB  (0.3s)
✓ test_applications            30,000 baris      3.5 MB →    1.0 MB  (0.1s)
✓ sample_submission            30,000 baris      0.8 MB →    0.3 MB  (0.0s)
✓ customer_profiles           100,000 baris      6.4 MB →    1.6 MB  (0.1s)
✓ previous_applications       201,984 baris     26.7 MB →    8.0 MB  (0.7s)
✓ repayment_history         1,361,341 baris    159.7 MB →   32.7 MB  (2.4s)
✓ account_behavior          1,084,418 baris    100.0 MB →   29.4 MB  (1.4s)
✓ bureau_observations         797,563 baris    114.7 MB →   31.8 MB  (1.8s)
✓ ecommerce_transactions    4,359,817 baris    528.8 MB →  122.9 MB  (12.1s)


table,source,rows,cols,csv_mb,parquet_mb,seconds,ratio
str,str,i64,i64,f64,f64,f64,f64
"""train_applications""","""train_applications.csv""",70000,10,8.2,2.5,0.3,0.305
"""test_applications""","""test_applications.csv""",30000,9,3.5,1.0,0.1,0.286
"""sample_submission""","""sample_submission.csv""",30000,2,0.8,0.3,0.0,0.375
"""customer_profiles""","""customer_profiles.csv""",100000,8,6.4,1.6,0.1,0.25
"""previous_applications""","""previous_applications.csv""",201984,11,26.7,8.0,0.7,0.3
"""repayment_history""","""repayment_history.csv""",1361341,9,159.7,32.7,2.4,0.205
"""account_behavior""","""account_behavior.csv""",1084418,10,100.0,29.4,1.4,0.294
"""bureau_observations""","""bureau_observations.csv""",797563,14,114.7,31.8,1.8,0.277
"""ecommerce_transactions""","""ecommerce_transactions.csv""",4359817,10,528.8,122.9,12.1,0.232


In [5]:
manifest = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "polars_version": pl.__version__,
    "ecommerce_source": SOURCES["ecommerce_transactions"].name,
    "ecommerce_is_sampled": ECOM_IS_SAMPLED,
    "tables": report,
}
(PQ_DIR / "_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Total CSV    :", round(report_df["csv_mb"].sum(), 1), "MB")
print("Total Parquet:", round(report_df["parquet_mb"].sum(), 1), "MB")
print("Manifest     :", PQ_DIR / "_manifest.json")

Total CSV    : 948.8 MB
Total Parquet: 230.2 MB
Manifest     : d:\LOMBA\LOGIKA\data\parquet\_manifest.json


## 4. Cek cepat: skema hasil & cara membaca

Pola baca yang akan dipakai di notebook berikutnya:
```python
pl.read_parquet(PQ_DIR / "train_applications.parquet")                       # tabel kecil → langsung
pl.scan_parquet(PQ_DIR / "ecommerce_transactions.parquet")                   # tabel besar → lazy
  .filter(...).group_by("customer_id").agg(...).collect()                    # hanya hasil agregat yang masuk RAM
pl.read_parquet(path, columns=["customer_id", "month", "closing_balance"])   # hanya kolom tertentu
df.to_pandas()                                                               # kalau butuh pandas / sklearn
```

In [6]:
for name in SCHEMAS:
    sch = pl.scan_parquet(PQ_DIR / f"{name}.parquet").collect_schema()
    print(f"\n{name}")
    for c, t in sch.items():
        print(f"   {c:<32} {t}")


train_applications
   application_id                   String
   customer_id                      String
   application_datetime             Datetime(time_unit='us', time_zone=None)
   decision_datetime                Datetime(time_unit='us', time_zone=None)
   loan_amount                      Int64
   term_months                      Int32
   annual_interest_rate             Float64
   proposed_monthly_installment     Int64
   loan_purpose                     String
   target                           Int8

test_applications
   application_id                   String
   customer_id                      String
   application_datetime             Datetime(time_unit='us', time_zone=None)
   decision_datetime                Datetime(time_unit='us', time_zone=None)
   loan_amount                      Int64
   term_months                      Int32
   annual_interest_rate             Float64
   proposed_monthly_installment     Int64
   loan_purpose                     String

sample_submis

In [7]:
pl.read_parquet(PQ_DIR / "repayment_history.parquet", n_rows=5)

loan_id,installment_id,customer_id,due_date,amount_due,amount_paid,last_payment_datetime,days_past_due,max_days_past_due
str,str,str,datetime[μs],i64,i64,datetime[μs],i32,i32
"""L_14fd333c77dfbe2c""","""I_1d48725bf1ce96b1""","""C_0000026281a8d761""",2025-02-16 13:17:00,589751,589751,2025-02-16 13:17:00,0,0
"""L_14fd333c77dfbe2c""","""I_8f85f0cf6966238b""","""C_0000026281a8d761""",2025-03-16 13:17:00,589751,589751,2025-03-16 13:17:00,0,0
"""L_14fd333c77dfbe2c""","""I_cff1055d0008227f""","""C_0000026281a8d761""",2025-04-16 13:17:00,589751,589751,2025-04-16 13:17:00,0,0
"""L_14fd333c77dfbe2c""","""I_fd0b4df439ee35cb""","""C_0000026281a8d761""",2025-05-16 13:17:00,589751,589751,2025-05-16 13:17:00,0,0
"""L_14fd333c77dfbe2c""","""I_3b77bac873f180d9""","""C_0000026281a8d761""",2025-06-16 13:17:00,589751,589751,2025-06-16 13:17:00,0,0
